# LeetCode 3120 — Count the Number of Special Characters I bằng Python

Link đề: https://leetcode.com/problems/count-the-number-of-special-characters-i/ · Độ khó: **Easy** · Chủ đề: Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def numberOfSpecialChars(self, word: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **chữ cái** được gọi là **đặc biệt** nếu nó xuất hiện **cả dạng chữ thường lẫn chữ hoa** trong chuỗi `word`; cần **đếm số chữ cái đặc biệt** (đếm **chữ cái khác nhau**, không phải số lần xuất hiện).
2. Nắm các công cụ Python: `in` trên chuỗi, `str.islower()` / `isupper()` / `lower()` / `upper()`, **`set`** (tập hợp, loại trùng, phép giao `&`), `string.ascii_lowercase`, và **bitmask** (`|=`, `1 << i`, `&`).
3. Đi từ cách **với mỗi chữ cái `a..z` hỏi "có cả thường và hoa không?"** → **hai tập (tập chữ thường, tập chữ hoa) rồi giao** → **một tập duy nhất + duyệt bảng chữ cái** → **hai mảng cờ `26` phần tử** → **hai bitmask `26` bit**.
4. Hiểu mấu chốt: chỉ cần biết **với mỗi chữ cái: đã thấy dạng thường chưa, đã thấy dạng hoa chưa** (hai thông tin boolean); số lần lặp **không quan trọng**.
5. Tránh các lỗi kinh điển: đếm theo **số lần xuất hiện** (đếm `a` hai lần), đếm `A` và `a` như hai chữ cái đặc biệt khác nhau, nhầm `ord` của chữ hoa/chữ thường (`65` vs `97`), và so sánh sai kiểu hoa-thường (`'a' == 'A'`).

## 2. Đề bài & ví dụ

Cho một chuỗi `word`. Một chữ cái được gọi là **đặc biệt** (special) nếu **nó xuất hiện cả ở dạng chữ thường lẫn dạng chữ hoa** trong `word`.

Hãy trả về **số chữ cái đặc biệt** (mỗi chữ cái đếm **một lần**) trong `word`.

**Ví dụ 1**

- Input: `word = "aaAbcBC"`
- Output: `3`
- Giải thích: các chữ cái đặc biệt là `a` (có `a` và `A`), `b` (có `b` và `B`), `c` (có `c` và `C`). Đáp án `3`.

**Ví dụ 2**

- Input: `word = "abc"`
- Output: `0`
- Giải thích: không có chữ cái nào xuất hiện ở dạng hoa.

**Ví dụ 3**

- Input: `word = "abBCab"`
- Output: `1`
- Giải thích: chỉ có `b` xuất hiện cả `b` lẫn `B`. (`C` chỉ có dạng hoa, `a` chỉ có dạng thường.)

**Ràng buộc:** `1 <= word.length <= 50`; `word` chỉ gồm chữ cái tiếng Anh **thường và hoa**.

In [1]:
import random
import string

examples = [
    ("aaAbcBC", 3),
    ("abc", 0),
    ("abBCab", 1),
]
for word, expected in examples:
    print(repr(word), "-> kỳ vọng", expected)

'aaAbcBC' -> kỳ vọng 3
'abc' -> kỳ vọng 0
'abBCab' -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Chữ thường và chữ hoa: `islower`, `isupper`, `lower`, `upper`

- `c.islower()` là `True` nếu `c` là chữ thường; `c.isupper()` nếu là chữ hoa.
- `c.lower()` đổi sang chữ thường (`'A'.lower()` là `'a'`); `c.upper()` đổi sang chữ hoa.
- `'a' == 'A'` là `False`: máy tính coi hai ký tự **khác nhau** (mã `97` và `65`).

In [2]:
print("a".islower(), "A".isupper(), "a".isupper())     # True True False
print("A".lower(), "a".upper())                        # a A
print("a" == "A", "a" == "A".lower())                  # False True
print(ord("a"), ord("A"), ord("a") - ord("A"))         # 97 65 32

True True False
a A
False True
97 65 32


### 3.2. Toán tử `in` trên chuỗi

`x in word` là `True` nếu ký tự (hoặc chuỗi con) `x` xuất hiện trong `word`. Cho chuỗi dài `n`, đây là phép tìm kiếm tuyến tính `O(n)`. `'a' in "aaAbcBC"` là `True`; `'A' in "abc"` là `False`.

In [3]:
word = "aaAbcBC"
print("a" in word, "A" in word, "C" in word, "z" in word)    # True True True False

True True True False


### 3.3. `set` (tập hợp) và phép giao `&`

`set` chứa các phần tử **không trùng nhau** và hỗ trợ kiểm tra `x in tập` rất nhanh (trung bình `O(1)`). Phép **giao** `a & b` là tập các phần tử có ở **cả hai** `a` và `b`. `len(tập)` đếm số phần tử khác nhau.

In [4]:
word = "aaAbcBC"
lowers = set(c for c in word if c.islower())       # {'a', 'b', 'c'}
uppers = set(c for c in word if c.isupper())       # {'A', 'B', 'C'}
print(lowers, uppers)
print(lowers & {c.lower() for c in uppers})        # giao sau khi đổi hoa -> thường: {'a', 'b', 'c'}
print(len({1, 1, 2, 2, 3}))                        # 3 (loại trùng)

{'a', 'c', 'b'} {'C', 'A', 'B'}
{'a', 'c', 'b'}
3


### 3.4. `string.ascii_lowercase` và duyệt bảng chữ cái

`string.ascii_lowercase` là chuỗi `'abcdefghijklmnopqrstuvwxyz'`, `string.ascii_uppercase` là `'ABC...Z'`. Với bài này ta có thể **duyệt 26 chữ cái** thay vì duyệt chuỗi `word`, rồi hỏi mỗi chữ cái "có cả hai dạng không".

In [5]:
print(string.ascii_lowercase)
print(string.ascii_uppercase)
print(len(string.ascii_lowercase))                 # 26

abcdefghijklmnopqrstuvwxyz
ABCDEFGHIJKLMNOPQRSTUVWXYZ
26


### 3.5. Bitmask: dùng một số nguyên làm "tập 26 phần tử"

Một số nguyên có thể coi như một **dãy bit**: bit thứ `i` bằng `1` nghĩa là "chữ cái thứ `i` đã xuất hiện". Ta cần:

- `1 << i`: số chỉ có bit `i` bật.
- `mask |= 1 << i`: **bật** bit `i` của `mask`.
- `mask & (1 << i)`: khác `0` nếu bit `i` đang bật.
- `a & b`: phép **giao** của hai bitmask (bit nào bật ở cả hai).
- Đếm số bit `1`: `bin(x).count("1")` (hoặc `x.bit_count()` ở Python 3.10+).

In [6]:
mask = 0
for i in [0, 2, 5]:
    mask |= 1 << i
print(bin(mask), mask)                             # 0b100101 37
print(bool(mask & (1 << 2)), bool(mask & (1 << 3)))   # True False
other = (1 << 2) | (1 << 3)
print(bin(mask & other), bin(mask & other).count("1"))   # 0b100 1

0b100101 37
True False
0b100 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `word = "aaAbcBC"`. Lập bảng cho từng chữ cái: đã thấy dạng thường? đã thấy dạng hoa?

| chữ cái | thấy dạng thường | thấy dạng hoa | đặc biệt? |
|---|---|---|---|
| a | có (`a`, `a`) | có (`A`) | **có** |
| b | có | có (`B`) | **có** |
| c | có | có (`C`) | **có** |
| các chữ khác | không | không | không |

Có 3 chữ cái đặc biệt. Để ý: số lần `a` lặp lại (2 lần) **không** ảnh hưởng; chữ cái chỉ cần **xuất hiện ít nhất một lần** ở mỗi dạng.

Với `"abBCab"`: `a` chỉ có dạng thường, `b` có cả `b` và `B` ⇒ đặc biệt; `C` chỉ có dạng hoa ⇒ không. Đáp án `1`.

In [7]:
word = "abBCab"
for ch in string.ascii_lowercase:
    low = ch in word
    up = ch.upper() in word
    if low or up:
        print(f"{ch}: có thường={low}, có hoa={up} -> {'đặc biệt' if low and up else 'không'}")

a: có thường=True, có hoa=False -> không
b: có thường=True, có hoa=True -> đặc biệt
c: có thường=False, có hoa=True -> không


## 5. Cách 1 — Brute Force / baseline: với mỗi chữ cái `a..z` hỏi bằng `in`

**Ý tưởng.** Duyệt từng chữ cái thường `ch` của bảng chữ cái; chữ cái đó đặc biệt nếu `ch in word` **và** `ch.upper() in word`. Đếm các chữ cái thỏa. Rất sát lời đề; mỗi `in` quét `word` một lần.

In [8]:
def count_special_alphabet(word: str) -> int:
    count = 0
    for ch in string.ascii_lowercase:                  # 26 chữ cái
        if ch in word and ch.upper() in word:          # có cả dạng thường lẫn dạng hoa
            count += 1
    return count


for word, expected in examples:
    print(repr(word), count_special_alphabet(word), "(kỳ vọng", expected, ")")

'aaAbcBC' 3 (kỳ vọng 3 )
'abc' 0 (kỳ vọng 0 )
'abBCab' 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Gọi `n = len(word)`. 26 chữ cái × hai lần `in` (mỗi lần `O(n)`) ⇒ **Time `O(26 · n)`**, tức `O(n)` với hằng số `52`; **Space `O(1)`**.

**Nút thắt:** mỗi lần `in` quét lại cả chuỗi. Ta có thể duyệt chuỗi **một lần** để ghi lại "đã thấy chữ nào, dạng nào" rồi trả lời nhanh bằng tập hợp.

## 6. Ý tưởng thuật toán / bước đột phá

Đề chỉ hỏi "đã thấy chữ cái này ở dạng thường / dạng hoa chưa" — **hai thông tin boolean** cho mỗi chữ cái. Vậy:

1. Duyệt `word` **một lần**, ghi nhận mỗi ký tự: ký tự thường ⇒ cho vào tập `lowers`; ký tự hoa ⇒ cho vào tập `uppers` (sau khi đổi về chữ thường để so sánh).
2. Số chữ cái đặc biệt = **số phần tử của giao** `lowers & uppers`.

Số lần lặp lại không còn ảnh hưởng vì `set` tự loại trùng.

In [9]:
word = "aaAbcBC"
lowers = {c for c in word if c.islower()}
uppers_as_lower = {c.lower() for c in word if c.isupper()}
print("lowers =", sorted(lowers), "| uppers (đã đổi sang thường) =", sorted(uppers_as_lower))
print("giao =", sorted(lowers & uppers_as_lower), "-> đáp án", len(lowers & uppers_as_lower))

lowers = ['a', 'b', 'c'] | uppers (đã đổi sang thường) = ['a', 'b', 'c']
giao = ['a', 'b', 'c'] -> đáp án 3


## 7. Các cách tiếp theo

### Cách 2 — Hai tập và phép giao

**Giải thích từng đoạn.**

1. `lowers`: tập các chữ thường xuất hiện trong `word`.
2. `uppers`: tập các chữ hoa xuất hiện, **đã đổi về chữ thường** (`c.lower()`) để so sánh được với `lowers`.
3. Đáp án `len(lowers & uppers)`.

In [10]:
def count_special_sets(word: str) -> int:
    lowers = {c for c in word if c.islower()}
    uppers = {c.lower() for c in word if c.isupper()}      # đổi về chữ thường để so khớp
    return len(lowers & uppers)


for word, expected in examples:
    print(repr(word), count_special_sets(word), "(kỳ vọng", expected, ")")

'aaAbcBC' 3 (kỳ vọng 3 )
'abc' 0 (kỳ vọng 0 )
'abBCab' 1 (kỳ vọng 1 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSets:
    def numberOfSpecialChars(self, word: str) -> int:
        lowers = {c for c in word if c.islower()}
        uppers = {c.lower() for c in word if c.isupper()}
        return len(lowers & uppers)


print(SolutionSets().numberOfSpecialChars("abBCab"))      # 1

1


**Độ phức tạp Cách 2:** **Time `O(n)`** (một lượt duyệt, mỗi lần thêm vào `set` trung bình `O(1)`), **Space `O(1)`** (tối đa 26 phần tử mỗi tập).

### Cách 3 — Một tập `set(word)` rồi duyệt bảng chữ cái

**Giải thích.** `seen = set(word)` chứa mọi ký tự **riêng biệt** của `word` (cả hoa lẫn thường, giữ nguyên dạng). Rồi duyệt 26 chữ cái: `ch in seen and ch.upper() in seen`. Mỗi kiểm tra `in` trên `set` là `O(1)`, nên tổng chỉ `O(n + 26)`.

In [12]:
def count_special_one_set(word: str) -> int:
    seen = set(word)                                              # các ký tự riêng biệt
    return sum(ch in seen and ch.upper() in seen for ch in string.ascii_lowercase)


for word, expected in examples:
    print(repr(word), count_special_one_set(word), "(kỳ vọng", expected, ")")

'aaAbcBC' 3 (kỳ vọng 3 )
'abc' 0 (kỳ vọng 0 )
'abBCab' 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(n + 26)`**, **Space `O(1)`** (tối đa 52 ký tự khác nhau). `sum` cộng các `True` như `1`.

### Cách 4 — Hai mảng cờ 26 phần tử

**Giải thích.** `has_lower[i]` = "chữ cái thứ `i` đã xuất hiện dạng thường", `has_upper[i]` = "... dạng hoa". Chỉ số: `ord(c) - ord('a')` cho chữ thường, `ord(c) - ord('A')` cho chữ hoa. Cuối cùng đếm các `i` có cả hai cờ. Cách này không dùng `set` và hay gặp trong bài về bảng chữ cái.

In [13]:
def count_special_flags(word: str) -> int:
    has_lower = [False] * 26
    has_upper = [False] * 26
    for c in word:
        if "a" <= c <= "z":
            has_lower[ord(c) - ord("a")] = True      # chữ thường: chỉ số theo 'a'
        else:
            has_upper[ord(c) - ord("A")] = True      # chữ hoa: chỉ số theo 'A'
    return sum(1 for i in range(26) if has_lower[i] and has_upper[i])


for word, expected in examples:
    print(repr(word), count_special_flags(word), "(kỳ vọng", expected, ")")

'aaAbcBC' 3 (kỳ vọng 3 )
'abc' 0 (kỳ vọng 0 )
'abBCab' 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n + 26)`**, **Space `O(26)`**.

### Cách 5 — Hai bitmask

**Giải thích.** Thay mỗi mảng cờ bằng **một số nguyên 26 bit**: `lower_mask`, `upper_mask`. Gặp chữ thường thứ `i` thì `lower_mask |= 1 << i`; gặp chữ hoa thứ `i` thì `upper_mask |= 1 << i`. Đáp án là số bit `1` của `lower_mask & upper_mask`.

In [14]:
def count_special_bitmask(word: str) -> int:
    lower_mask = upper_mask = 0
    for c in word:
        if c.islower():
            lower_mask |= 1 << (ord(c) - ord("a"))
        else:
            upper_mask |= 1 << (ord(c) - ord("A"))
    return bin(lower_mask & upper_mask).count("1")       # số bit 1 = số chữ cái có cả hai dạng


for word, expected in examples:
    print(repr(word), count_special_bitmask(word), "(kỳ vọng", expected, ")")

'aaAbcBC' 3 (kỳ vọng 3 )
'abc' 0 (kỳ vọng 0 )
'abBCab' 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (hai số nguyên). Gọn nhất về bộ nhớ.

## 8. TRACE từng bước

Ta trace Cách 2 (hai tập) và Cách 5 (bitmask) trên `"abBCab"`.

In [15]:
def trace_sets(word: str) -> int:
    lowers, uppers = set(), set()
    print("word =", word)
    for i, c in enumerate(word):
        if c.islower():
            lowers.add(c)
        else:
            uppers.add(c.lower())
        print(f"i={i} c={c!r}: lowers={sorted(lowers)} uppers(thường)={sorted(uppers)}")
    both = lowers & uppers
    print("giao =", sorted(both), "-> đáp án", len(both))
    return len(both)


assert trace_sets("abBCab") == 1

word = abBCab
i=0 c='a': lowers=['a'] uppers(thường)=[]
i=1 c='b': lowers=['a', 'b'] uppers(thường)=[]
i=2 c='B': lowers=['a', 'b'] uppers(thường)=['b']
i=3 c='C': lowers=['a', 'b'] uppers(thường)=['b', 'c']
i=4 c='a': lowers=['a', 'b'] uppers(thường)=['b', 'c']
i=5 c='b': lowers=['a', 'b'] uppers(thường)=['b', 'c']
giao = ['b'] -> đáp án 1


In [16]:
def trace_bitmask(word: str) -> int:
    lower_mask = upper_mask = 0
    for i, c in enumerate(word):
        bit = ord(c.lower()) - ord("a")
        if c.islower():
            lower_mask |= 1 << bit
        else:
            upper_mask |= 1 << bit
        print(f"i={i} c={c!r} (bit {bit}): lower={lower_mask:026b} upper={upper_mask:026b}")
    both = lower_mask & upper_mask
    print(f"AND = {both:026b} -> {bin(both).count('1')} bit 1")
    return bin(both).count("1")


assert trace_bitmask("abBCab") == 1

i=0 c='a' (bit 0): lower=00000000000000000000000001 upper=00000000000000000000000000
i=1 c='b' (bit 1): lower=00000000000000000000000011 upper=00000000000000000000000000
i=2 c='B' (bit 1): lower=00000000000000000000000011 upper=00000000000000000000000010
i=3 c='C' (bit 2): lower=00000000000000000000000011 upper=00000000000000000000000110
i=4 c='a' (bit 0): lower=00000000000000000000000011 upper=00000000000000000000000110
i=5 c='b' (bit 1): lower=00000000000000000000000011 upper=00000000000000000000000110
AND = 00000000000000000000000010 -> 1 bit 1


Bảng trace Cách 2 với `"abBCab"`:

| i | c | lowers | uppers (chữ thường) |
|---|---|---|---|
| 0 | a | {a} | {} |
| 1 | b | {a, b} | {} |
| 2 | B | {a, b} | {b} |
| 3 | C | {a, b} | {b, c} |
| 4 | a | {a, b} | {b, c} |
| 5 | b | {a, b} | {b, c} |

Giao `{a, b} ∩ {b, c} = {b}` ⇒ đáp án `1`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (hai tập và giao): ngắn, rõ nghĩa, `O(n)`.

In [17]:
class Solution:
    def numberOfSpecialChars(self, word: str) -> int:
        lowers = {c for c in word if c.islower()}              # các chữ thường đã xuất hiện
        uppers = {c.lower() for c in word if c.isupper()}      # các chữ hoa đã xuất hiện (đổi về thường)
        return len(lowers & uppers)                            # chữ cái có cả hai dạng


sol = Solution()
for word, expected in examples:
    print(repr(word), "->", sol.numberOfSpecialChars(word))

'aaAbcBC' -> 3
'abc' -> 0
'abBCab' -> 1


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa:** chữ cái `x` đặc biệt ⇔ `x` (thường) ∈ `word` **và** `x.upper()` ∈ `word`. Tập `lowers` chứa mọi chữ thường có trong `word`; tập `uppers` (đã đổi về thường) chứa mọi `x` mà `x.upper()` có trong `word`. Vậy `x ∈ lowers ∩ uppers` ⇔ `x` đặc biệt.
- **Đếm mỗi chữ cái một lần:** `len` của một `set` đếm **phần tử khác nhau**, nên chữ cái lặp lại nhiều lần chỉ tính một lần.
- **Không đếm trùng hoa/thường:** ta đổi chữ hoa về chữ thường **trước khi** giao; nếu không, `'a'` và `'A'` là hai phần tử khác nhau và giao sẽ rỗng.
- **Các cách khác tương đương:** Cách 1, 3 trực tiếp hỏi hai điều kiện trên; Cách 4, 5 chỉ biểu diễn hai tập bằng mảng cờ / bitmask.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word)`.

| Cách | Time | Space |
|---|---|---|
| 1. 26 chữ cái × `in` | `O(26 · n)` | `O(1)` |
| 2. Hai tập và giao | `O(n)` | `O(1)` |
| 3. `set(word)` + 26 chữ cái | `O(n + 26)` | `O(1)` |
| 4. Hai mảng cờ | `O(n + 26)` | `O(26)` |
| 5. Hai bitmask | `O(n)` | `O(1)` |

(Không gian `O(1)` vì tập chứa tối đa 52 ký tự — hằng số.)

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "alphabet": count_special_alphabet,
    "sets": count_special_sets,
    "one_set": count_special_one_set,
    "flags": count_special_flags,
    "bitmask": count_special_bitmask,
    "leetcode": sol.numberOfSpecialChars,
}


def check_all(word, expected, label):
    for key, fn in impls.items():
        actual = fn(word)
        assert actual == expected, (label, key, word, actual, expected)
    print("PASS", label)


for idx, (word, expected) in enumerate(examples, 1):
    check_all(word, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all("a", 0, "một chữ thường")
check_all("A", 0, "một chữ hoa")
check_all("aA", 1, "một cặp thường-hoa")
check_all("Aa", 1, "thứ tự hoa-thường")
check_all("aa", 0, "lặp lại chữ thường")
check_all("AA", 0, "lặp lại chữ hoa")
check_all("aaAAaaAA", 1, "lặp nhiều lần vẫn tính một chữ cái")
check_all("abcABC", 3, "ba chữ cái đặc biệt")
check_all("abcdef", 0, "toàn chữ thường")
check_all("ABCDEF", 0, "toàn chữ hoa")
check_all("aBcD", 0, "mỗi chữ cái chỉ có một dạng")
check_all("aAbBcCdDeEfFgGhHiIjJkKlLmMnNoOpPqQrRsStTuUvVwWxXyYzZ", 26, "đủ 26 chữ cái đặc biệt (52 ký tự)")
check_all("z" * 25 + "Z" * 25, 1, "n = 50: chỉ chữ z")
check_all("aAbc", 1, "một đặc biệt, hai chỉ thường")

PASS một chữ thường
PASS một chữ hoa
PASS một cặp thường-hoa
PASS thứ tự hoa-thường
PASS lặp lại chữ thường
PASS lặp lại chữ hoa
PASS lặp nhiều lần vẫn tính một chữ cái
PASS ba chữ cái đặc biệt
PASS toàn chữ thường
PASS toàn chữ hoa
PASS mỗi chữ cái chỉ có một dạng
PASS đủ 26 chữ cái đặc biệt (52 ký tự)
PASS n = 50: chỉ chữ z
PASS một đặc biệt, hai chỉ thường


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **đếm số lần xuất hiện** thay vì số chữ cái; (b) **không đổi chữ hoa về chữ thường trước khi giao** (giao hai tập mà một tập là chữ hoa, tập kia là chữ thường).

In [20]:
def wrong_count_occurrences(word):
    return sum(c.islower() and c.upper() in word for c in word)      # đếm cả lần lặp


def wrong_no_lower(word):
    lowers = {c for c in word if c.islower()}
    uppers = {c for c in word if c.isupper()}                         # SAI: chưa đổi về thường
    return len(lowers & uppers)


bad1 = "aaAbcBC"
bad2 = "aA"
print(wrong_count_occurrences(bad1), "!=", count_special_sets(bad1))
print(wrong_no_lower(bad2), "!=", count_special_sets(bad2))
assert wrong_count_occurrences(bad1) != count_special_sets(bad1)
assert wrong_no_lower(bad2) != count_special_sets(bad2)

4 != 3
0 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi đúng ràng buộc (độ dài `1..50`, chỉ chữ cái hoa/thường). Tham chiếu độc lập: so sánh từng chữ cái bằng `word.count`.

In [21]:
def reference(word):
    return sum(1 for ch in string.ascii_lowercase if word.count(ch) > 0 and word.count(ch.upper()) > 0)


rng = random.Random(3120)
for trial in range(3000):
    alphabet_size = rng.choice([1, 2, 3, 5, 10, 26])
    letters = string.ascii_lowercase[:alphabet_size]
    pool = letters + letters.upper()
    n = rng.choice([1, 2, 3, 5, 10, 25, 50])
    word = "".join(rng.choice(pool) for _ in range(n))
    assert 1 <= len(word) <= 50 and word.isalpha()
    ref = reference(word)
    for key, fn in impls.items():
        assert fn(word) == ref, (key, word)
print("PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc: mọi cách khớp tham chiếu")

PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc: mọi cách khớp tham chiếu


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số lần xuất hiện thay vì số chữ cái

In [22]:
def sai_dem_lan(word):
    count = 0
    for c in word:
        if c.islower() and c.upper() in word:       # SAI: mỗi lần lặp của 'a' đều đếm
            count += 1
    return count


print("sai:", sai_dem_lan("aaAbcBC"), "| đúng:", Solution().numberOfSpecialChars("aaAbcBC"))

sai: 4 | đúng: 3


**Vì sao sai:** đề hỏi **số chữ cái** đặc biệt (mỗi chữ cái một lần). `a` xuất hiện hai lần ở dạng thường nên bị đếm hai lần (`4` thay vì `3`). **Cách sửa:** dùng `set` (tự loại trùng) hoặc duyệt bảng chữ cái.

### Lỗi 2: Giao hai tập mà chưa đổi hoa thành thường

In [23]:
def sai_khong_doi_hoa(word):
    lowers = {c for c in word if c.islower()}
    uppers = {c for c in word if c.isupper()}       # {'A'}, không khớp với 'a'
    return len(lowers & uppers)


print("sai:", sai_khong_doi_hoa("aA"), "| đúng:", Solution().numberOfSpecialChars("aA"))

sai: 0 | đúng: 1


**Vì sao sai:** `'a'` và `'A'` là hai ký tự khác nhau nên giao `{'a'} & {'A'}` rỗng. **Cách sửa:** đổi chữ hoa về chữ thường (`c.lower()`) trước khi giao, hoặc đổi chữ thường về hoa.

### Lỗi 3: Nhầm công thức chỉ số `ord` giữa chữ hoa và chữ thường

In [24]:
def sai_ord(word):
    has_lower = [False] * 26
    has_upper = [False] * 26
    for c in word:
        if c.islower():
            has_lower[ord(c) - ord("a")] = True
        else:
            has_upper[ord(c) - ord("a")] = True       # SAI: chữ hoa phải trừ ord('A')
    return sum(1 for i in range(26) if has_lower[i] and has_upper[i])


try:
    print("sai:", sai_ord("aA"), "| đúng:", Solution().numberOfSpecialChars("aA"))
except IndexError as e:
    print("sai: IndexError:", e, "| đúng:", Solution().numberOfSpecialChars("aA"))

sai: IndexError: list assignment index out of range | đúng: 1


**Vì sao sai:** `ord('A') - ord('a') = -32`: chỉ số âm **nhỏ hơn `-26`** nên `has_upper[-32]` báo `IndexError`. Tệ hơn, với chữ hoa khác (ví dụ `'Z'`: `ord('Z') - ord('a') = -7`) chỉ số âm nằm trong khoảng hợp lệ nên Python **không báo lỗi** mà ghi cờ vào vị trí `26 - 7 = 19` — một chữ cái hoàn toàn khác. **Cách sửa:** chữ hoa dùng `ord(c) - ord("A")`; hoặc đổi `c.lower()` trước rồi trừ `ord("a")`.

### Lỗi 4: So sánh trực tiếp hoa và thường

In [25]:
def sai_so_sanh(word):
    count = 0
    for ch in string.ascii_lowercase:
        if ch in word and ch in word:               # SAI: viết cùng một điều kiện hai lần (quên .upper())
            count += 1
    return count


print("sai:", sai_so_sanh("abc"), "| đúng:", Solution().numberOfSpecialChars("abc"))

sai: 3 | đúng: 0


**Vì sao sai:** điều kiện thứ hai phải hỏi chữ **hoa** (`ch.upper() in word`). Viết nhầm khiến mọi chữ thường đều bị coi là đặc biệt. **Cách sửa:** `ch in word and ch.upper() in word`. Mẹo: sau khi viết điều kiện, thử một input chỉ có chữ thường (`"abc"`) và kỳ vọng `0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chữ cái xuất hiện ở **cả hai dạng/hai nhóm**" ⇒ hai `set` rồi **giao** (hoặc hai bitmask rồi `&`).
- Đếm **số phần tử khác nhau** ⇒ dùng `set` và `len`; đừng đếm theo số lần.
- Bảng chữ cái cố định `26` ⇒ mảng cờ hoặc bitmask `O(1)` bộ nhớ.
- Chữ hoa ↔ chữ thường: `c.lower()` / `c.upper()`; hoặc khoảng cách mã `32` giữa `'a'` và `'A'`.
- Luôn thử `"aA"`, `"a"`, `"aaAA"` để kiểm tra "đếm một lần".

## 15. Biến thể của bài

1. **Count the Number of Special Characters II (3121):** chữ cái đặc biệt nếu **mọi** chữ thường xuất hiện **trước** chữ hoa đầu tiên (thứ tự quan trọng).
2. **Trả về danh sách các chữ cái đặc biệt**, sắp xếp.
3. **Chữ cái chỉ xuất hiện một dạng** (hoặc chỉ thường, hoặc chỉ hoa).
4. **Đếm số cặp thường-hoa** ở vị trí liền kề (`aA`).
5. **Greatest English Letter in Upper and Lower Case (2309)**: chữ cái lớn nhất có cả hai dạng.

In [26]:
def special_letters(word):
    return sorted({c for c in word if c.islower()} & {c.lower() for c in word if c.isupper()})


def only_one_form(word):
    lowers = {c for c in word if c.islower()}
    uppers = {c.lower() for c in word if c.isupper()}
    return sorted(lowers ^ uppers)                   # ^: đối xứng hiệu = có ở đúng một tập


def greatest_both(word):
    both = special_letters(word)
    return both[-1].upper() if both else ""


print(special_letters("aaAbcBC"))        # ['a', 'b', 'c']
print(only_one_form("abBCab"))           # ['a', 'c']
print(greatest_both("lEeTcOdE"), repr(greatest_both("abc")))
assert special_letters("aaAbcBC") == ["a", "b", "c"]
assert only_one_form("abBCab") == ["a", "c"]
assert greatest_both("lEeTcOdE") == "E"
assert greatest_both("abc") == ""

['a', 'b', 'c']
['a', 'c']
E ''


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. 26 chữ cái × `in` | hỏi từng chữ cái | `O(26 · n)` | `O(1)` | dễ hiểu nhất |
| 2. Hai tập và giao | `lowers & uppers` | `O(n)` | `O(1)` | **nên nộp**, đúng bản chất |
| 3. `set(word)` + 26 chữ cái | tra tập `O(1)` | `O(n + 26)` | `O(1)` | khi thích duyệt bảng chữ cái |
| 4. Hai mảng cờ | `has_lower`, `has_upper` | `O(n + 26)` | `O(26)` | luyện chỉ số `ord` |
| 5. Hai bitmask | bật bit, rồi AND và đếm bit | `O(n)` | `O(1)` | cần tối ưu bộ nhớ/tốc độ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"Q".lower()`, `"aBc".swapcase()`, `{c for c in "aAa"}`, `Solution().numberOfSpecialChars("xXyY")`, `Solution().numberOfSpecialChars("xxXY")`.

**Bài 2 (tính tay).** Với `word = "AbaBcCdD"`: lập bảng `chữ cái / có thường? / có hoa?` và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thành `special_todo(word)` theo Cách 2.

In [27]:
def special_todo(word):
    # TODO 1: lowers = tập các chữ thường trong word
    # TODO 2: uppers = tập các chữ hoa trong word, ĐÃ đổi về chữ thường
    # TODO 3: trả về số phần tử của giao lowers và uppers
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `special_bitmask2(word)` theo Cách 5 nhưng đếm bit bằng **vòng `while mask: mask &= mask - 1`** (xóa bit thấp nhất) thay cho `bin(...).count("1")`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `special_ii(word)` cho bài **3121**: chữ cái `c` đặc biệt nếu `c` xuất hiện cả hai dạng và **mọi chữ thường `c` đều đứng trước mọi chữ hoa `C`** (ví dụ `"aaAbcBC"` → `3`, `"abBCab"` → `0` vì `b` thường còn xuất hiện sau `B`). (b) Viết `only_lowercase_letters(word)` trả về danh sách các chữ cái **chỉ** xuất hiện dạng thường, sắp xếp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'q'`, `'AbC'`, `{'a', 'A'}`, `2`, `1`. Bài 3: `{c for c in word if c.islower()}` và `{c.lower() for c in word if c.isupper()}`.
- **Mức 2 (vừa):** Bài 2: các ký tự là `A, b, a, B, c, C, d, D`: `a` có `a` và `A`; `b` có `b` và `B`; `c` có `c` và `C`; `d` có `d` và `D` ⇒ cả `4` chữ cái đều đặc biệt. Bài 4: `while mask: mask &= mask - 1; count += 1`.
- **Mức 3 (sâu):** Bài 5(a): với mỗi chữ cái `c`, lấy chỉ số **cuối cùng** của chữ thường `c` (`word.rfind(c)`) và chỉ số **đầu tiên** của chữ hoa `C` (`word.find(C)`); chữ cái đặc biệt nếu cả hai tồn tại và `rfind(c) < find(C)`. 5(b): `sorted(lowers - uppers)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "Q".lower() == "q"
assert "aBc".swapcase() == "AbC"
assert {c for c in "aAa"} == {"a", "A"}
assert Solution().numberOfSpecialChars("xXyY") == 2
assert Solution().numberOfSpecialChars("xxXY") == 1

# Bài 2
assert Solution().numberOfSpecialChars("AbaBcCdD") == 4


# Bài 3
def special_todo_ans(word):
    lowers = {c for c in word if c.islower()}
    uppers = {c.lower() for c in word if c.isupper()}
    return len(lowers & uppers)


for word, expected in examples:
    assert special_todo_ans(word) == expected


# Bài 4
def special_bitmask2(word):
    lower_mask = upper_mask = 0
    for c in word:
        if c.islower():
            lower_mask |= 1 << (ord(c) - ord("a"))
        else:
            upper_mask |= 1 << (ord(c) - ord("A"))
    mask = lower_mask & upper_mask
    count = 0
    while mask:
        mask &= mask - 1
        count += 1
    return count


for word, expected in examples:
    assert special_bitmask2(word) == expected


# Bài 5(a)
def special_ii(word):
    count = 0
    for c in string.ascii_lowercase:
        last_lower = word.rfind(c)
        first_upper = word.find(c.upper())
        if last_lower != -1 and first_upper != -1 and last_lower < first_upper:
            count += 1
    return count


assert special_ii("aaAbcBC") == 3
assert special_ii("abBCab") == 0
assert special_ii("abc") == 0
assert special_ii("AbBCab") == 0


# Bài 5(b)
def only_lowercase_letters(word):
    lowers = {c for c in word if c.islower()}
    uppers = {c.lower() for c in word if c.isupper()}
    return sorted(lowers - uppers)


assert only_lowercase_letters("abBCab") == ["a"]
assert only_lowercase_letters("aaAbcBC") == []
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chữ cái xuất hiện ở cả hai nhóm ⇒ hai tập, giao)

```python
class Solution:
    def numberOfSpecialChars(self, word: str) -> int:
        lowers = {c for c in word if c.islower()}
        uppers = {c.lower() for c in word if c.isupper()}
        return len(lowers & uppers)
```

**Checklist**

- [ ] Đếm **chữ cái khác nhau** (dùng `set`), không đếm số lần xuất hiện?
- [ ] Đã đổi chữ hoa về chữ thường (hoặc ngược lại) trước khi giao hai tập?
- [ ] Chữ cái chỉ có một dạng (chỉ thường hoặc chỉ hoa) **không** được đếm?
- [ ] Nếu dùng chỉ số `ord`: chữ hoa trừ `ord("A")`, chữ thường trừ `ord("a")`?
- [ ] Đã thử `"aA"`, `"aaAA"`, `"abc"`, `"aBcD"`?
- [ ] Skeleton đúng `class Solution` / `def numberOfSpecialChars(self, word: str) -> int`?